# Single-Column ParFlow-CLM Simulation

Run a single-column PF-CLM simulation with improved CLM physics:
- **Medlyn stomatal conductance** — VPD-based stomata (replaces Ball-Berry)
- **CLM5 tanh canopy interception** — physically-based interception scheme
- **Canopy clumping** — He et al. 2012 PFT-dependent clumping indices

This notebook uses the forcing and CLM inputs prepared in Notebook 1.
Output is written as NetCDF for analysis in Notebook 3.

**Requires:** ParFlow v3.14.1+ with CLM ET improvements (`feature/clm_et` branch or later).

In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from datetime import datetime

from helpers import CONUS2_SOILS

# Set PARFLOW_DIR before importing parflow
os.environ["PARFLOW_DIR"] = os.environ.get(
    "PARFLOW_DIR", str(Path.home() / "parflow" / "parflow_21_Jan_26"))
print(f"PARFLOW_DIR: {os.environ['PARFLOW_DIR']}")

from parflow import Run

In [ ]:
# ============================================================
# User Configuration — must match Notebook 1
# ============================================================
site_id    = "US-Slt"
start      = "2011-10-01"
end        = "2012-10-01"
water_year = 2012

static_write_dir = "/Users/reed/Projects/single_column_PFCLM/runs/US_Slt_WY2012"

# Forcing filename (written by Notebook 1)
forcing_filename = f"forcing1D.{site_id}.{start}-{end}.txt"

# Hours in water year
dt_start = datetime.strptime(start, "%Y-%m-%d")
dt_end = datetime.strptime(end, "%Y-%m-%d")
n_hours = int((dt_end - dt_start).total_seconds() / 3600)
print(f"Simulation: {site_id} WY{water_year}, {n_hours} hours")

In [ ]:
# ============================================================
# Soil Parameters
# ============================================================
# Look up from CONUS2 soil type (queried in Notebook 1), or set manually.
# Tuple: (name, ksat [m/h], porosity, vg_alpha [1/m], vg_n, sres)

soil_type = 1  # From Notebook 1's pf_indicator query
soil_info = CONUS2_SOILS[soil_type]
print(f"Soil type {soil_type}: {soil_info[0]}")

# Unpack — edit these directly if you want to override
ksat     = soil_info[1]   # Saturated hydraulic conductivity [m/h]
porosity = soil_info[2]   # Porosity [-]
vg_alpha = soil_info[3]   # Van Genuchten alpha [1/m]
vg_n     = soil_info[4]   # Van Genuchten n [-]
sres     = soil_info[5]   # Residual saturation [-]

# Water table depth BC (negative = below surface)
# Use value from Notebook 1's CONUS2/Ma2025 WTD query
wtd_m = -0.2  # meters (ParFlow BC convention)

print(f"  Ksat={ksat} m/h, porosity={porosity}")
print(f"  VG alpha={vg_alpha}, n={vg_n}, Sres={sres}")
print(f"  WTD BC={wtd_m} m")

In [ ]:
# ============================================================
# ParFlow-CLM Run Setup
# ============================================================
run_name = "pfclm_sc"
run = Run(run_name)
run.FileVersion = 4

# ----- TIMING -----
run.TimingInfo.BaseUnit     = 1.0
run.TimingInfo.StartCount   = 0
run.TimingInfo.StartTime    = 0.0
run.TimingInfo.StopTime     = n_hours
run.TimingInfo.DumpInterval = 1.0
run.TimeStep.Type           = "Constant"
run.TimeStep.Value          = 1.0

# ----- GRID (single column, 10 layers, 8m deep) -----
run.Process.Topology.P = 1
run.Process.Topology.Q = 1
run.Process.Topology.R = 1

run.ComputationalGrid.Lower.X = 0.0
run.ComputationalGrid.Lower.Y = 0.0
run.ComputationalGrid.Lower.Z = 0.0
run.ComputationalGrid.DX      = 2.0
run.ComputationalGrid.DY      = 2.0
run.ComputationalGrid.DZ      = 1.0
run.ComputationalGrid.NX      = 1
run.ComputationalGrid.NY      = 1
run.ComputationalGrid.NZ      = 10

# ----- DOMAIN -----
run.GeomInput.Names                  = "domain_input"
run.GeomInput.domain_input.InputType = "Box"
run.GeomInput.domain_input.GeomName  = "domain"

run.Geom.domain.Lower.X = 0.0
run.Geom.domain.Lower.Y = 0.0
run.Geom.domain.Lower.Z = 0.0
run.Geom.domain.Upper.X = 2.0
run.Geom.domain.Upper.Y = 2.0
run.Geom.domain.Upper.Z = 10.0
run.Geom.domain.Patches = "x_lower x_upper y_lower y_upper z_lower z_upper"

# ----- VARIABLE DZ -----
# Layers 0-6: 1.0m, layer 7: 0.6m, layer 8: 0.3m, layer 9 (top): 0.1m
# Total depth: 7*1.0 + 0.6 + 0.3 + 0.1 = 8.0m
run.Solver.Nonlinear.VariableDz = True
run.dzScale.GeomNames           = "domain"
run.dzScale.Type                = "nzList"
run.dzScale.nzListNumber        = 10

run.Cell._0.dzScale.Value = 1.0   # bottom
run.Cell._1.dzScale.Value = 1.0
run.Cell._2.dzScale.Value = 1.0
run.Cell._3.dzScale.Value = 1.0
run.Cell._4.dzScale.Value = 1.0
run.Cell._5.dzScale.Value = 1.0
run.Cell._6.dzScale.Value = 1.0
run.Cell._7.dzScale.Value = 0.6   # 60 cm
run.Cell._8.dzScale.Value = 0.3   # 30 cm
run.Cell._9.dzScale.Value = 0.1   # 10 cm (top)

# ----- HYDRAULIC PROPERTIES (from CONUS2 soil type) -----
run.Geom.Perm.Names              = "domain"
run.Geom.domain.Perm.Type        = "Constant"
run.Geom.domain.Perm.Value       = ksat
run.Perm.TensorType              = "TensorByGeom"
run.Geom.Perm.TensorByGeom.Names = "domain"
run.Geom.domain.Perm.TensorValX  = 1.0
run.Geom.domain.Perm.TensorValY  = 1.0
run.Geom.domain.Perm.TensorValZ  = 1.0

run.SpecificStorage.Type                       = "Constant"
run.SpecificStorage.GeomNames                  = "domain"
run.Geom.domain.SpecificStorage.Value          = 1.0e-4

run.Geom.Porosity.GeomNames    = "domain"
run.Geom.domain.Porosity.Type  = "Constant"
run.Geom.domain.Porosity.Value = porosity

# ----- PHASES -----
run.Phase.Names                    = "water"
run.Phase.water.Density.Type       = "Constant"
run.Phase.water.Density.Value      = 1.0
run.Phase.water.Viscosity.Type     = "Constant"
run.Phase.water.Viscosity.Value    = 1.0
run.Phase.water.Mobility.Type      = "Constant"
run.Phase.water.Mobility.Value     = 1.0

# Van Genuchten relative permeability
run.Phase.RelPerm.Type             = "VanGenuchten"
run.Phase.RelPerm.GeomNames        = "domain"
run.Geom.domain.RelPerm.Alpha      = vg_alpha
run.Geom.domain.RelPerm.N          = vg_n

# Van Genuchten saturation
run.Phase.Saturation.Type          = "VanGenuchten"
run.Phase.Saturation.GeomNames     = "domain"
run.Geom.domain.Saturation.Alpha   = vg_alpha
run.Geom.domain.Saturation.N       = vg_n
run.Geom.domain.Saturation.SRes    = sres
run.Geom.domain.Saturation.SSat    = 1.0

# ----- MISC -----
run.Contaminants.Names  = ""
run.Gravity             = 1.0
run.Domain.GeomName     = "domain"
run.Wells.Names         = ""
run.KnownSolution       = "NoKnownSolution"

# ----- TIME CYCLES -----
run.Cycle.Names                    = "constant"
run.Cycle.constant.Names           = "alltime"
run.Cycle.constant.alltime.Length   = 1
run.Cycle.constant.Repeat          = -1

# ----- BOUNDARY CONDITIONS -----
run.BCPressure.PatchNames = "x_lower x_upper y_lower y_upper z_lower z_upper"

run.Patch.x_lower.BCPressure.Type          = "FluxConst"
run.Patch.x_lower.BCPressure.Cycle         = "constant"
run.Patch.x_lower.BCPressure.alltime.Value = 0.0

run.Patch.y_lower.BCPressure.Type          = "FluxConst"
run.Patch.y_lower.BCPressure.Cycle         = "constant"
run.Patch.y_lower.BCPressure.alltime.Value = 0.0

# Bottom BC: equilibrium water table
run.Patch.z_lower.BCPressure.Type          = "DirEquilRefPatch"
run.Patch.z_lower.BCPressure.RefGeom       = "domain"
run.Patch.z_lower.BCPressure.RefPatch      = "z_upper"
run.Patch.z_lower.BCPressure.Cycle         = "constant"
run.Patch.z_lower.BCPressure.alltime.Value = wtd_m

run.Patch.x_upper.BCPressure.Type          = "FluxConst"
run.Patch.x_upper.BCPressure.Cycle         = "constant"
run.Patch.x_upper.BCPressure.alltime.Value = 0.0

run.Patch.y_upper.BCPressure.Type          = "FluxConst"
run.Patch.y_upper.BCPressure.Cycle         = "constant"
run.Patch.y_upper.BCPressure.alltime.Value = 0.0

run.Patch.z_upper.BCPressure.Type          = "OverlandFlow"
run.Patch.z_upper.BCPressure.Cycle         = "constant"
run.Patch.z_upper.BCPressure.alltime.Value = 0.0

# ----- SLOPES -----
run.TopoSlopesX.Type              = "Constant"
run.TopoSlopesX.GeomNames         = "domain"
run.TopoSlopesX.Geom.domain.Value = 0.1

run.TopoSlopesY.Type              = "Constant"
run.TopoSlopesY.GeomNames         = "domain"
run.TopoSlopesY.Geom.domain.Value = 0.0

run.Mannings.Type               = "Constant"
run.Mannings.GeomNames          = "domain"
run.Mannings.Geom.domain.Value  = 2.e-6

run.PhaseSources.water.Type              = "Constant"
run.PhaseSources.water.GeomNames         = "domain"
run.PhaseSources.water.Geom.domain.Value = 0.0

# ----- SOLVER -----
run.Solver         = "Richards"
run.Solver.MaxIter = 9000

run.Solver.Nonlinear.MaxIter           = 100
run.Solver.Nonlinear.ResidualTol       = 1e-5
run.Solver.Nonlinear.EtaChoice         = "Walker1"
run.Solver.Nonlinear.EtaValue          = 0.01
run.Solver.Nonlinear.UseJacobian       = True
run.Solver.Nonlinear.DerivativeEpsilon = 1e-12
run.Solver.Nonlinear.StepTol           = 1e-30
run.Solver.Nonlinear.Globalization     = "LineSearch"
run.Solver.Linear.KrylovDimension      = 100
run.Solver.Linear.MaxRestarts          = 5
run.Solver.Linear.Preconditioner       = "PFMG"
run.Solver.PrintSubsurf                = False
run.Solver.Drop                        = 1E-20
run.Solver.AbsTol                      = 1E-9

# ----- OUTPUT: PFB (disabled) -----
run.Solver.PrintSubsurfData    = False
run.Solver.PrintPressure       = False
run.Solver.PrintSaturation     = False
run.Solver.PrintCLM            = False
run.Solver.PrintMask           = False
run.Solver.PrintSpecificStorage = False
run.Solver.PrintEvapTrans      = False
run.Solver.WriteSiloMannings   = False
run.Solver.WriteSiloMask       = False
run.Solver.WriteSiloSlopes     = False
run.Solver.WriteSiloSaturation = False

# ----- OUTPUT: NetCDF (all output in single files) -----
run.NetCDF.NumStepsPerFile     = n_hours
run.NetCDF.WritePressure       = True
run.NetCDF.WriteSubsurface     = False
run.NetCDF.WriteSaturation     = True
run.NetCDF.WriteCLM            = True
run.NetCDF.CLMNumStepsPerFile  = n_hours

# ----- CLM CONFIGURATION -----
run.Solver.LSM                  = "CLM"
run.Solver.CLM.MetForcing       = "1D"
run.Solver.CLM.MetFileName      = forcing_filename
run.Solver.CLM.MetFilePath      = "."

run.Solver.CLM.EvapBeta         = "Linear"
run.Solver.CLM.VegWaterStress   = "Saturation"
run.Solver.CLM.ResSat           = 0.2
run.Solver.CLM.WiltingPoint     = 0.2
run.Solver.CLM.FieldCapacity    = 1.0
run.Solver.CLM.IrrigationType   = "none"
run.Solver.CLM.RZWaterStress    = 1
run.Solver.CLM.RootZoneNZ       = 5
run.Solver.CLM.SoiLayer         = 4

# ----- IMPROVED CLM PHYSICS -----
# Medlyn stomatal conductance (VPD-based, replaces Ball-Berry)
run.Solver.CLM.StomataScheme       = "Medlyn"
# CLM5 tanh canopy interception (replaces CLM3 fpi formula)
run.Solver.CLM.InterceptionScheme    = "CLM5Tanh"
run.Solver.CLM.InterceptionTanhAlpha = 1.0

# ----- CLM OUTPUT -----
run.Solver.PrintLSMSink         = False
run.Solver.CLM.CLMDumpInterval   = 1
run.Solver.CLM.CLMFileDir        = "output/"
run.Solver.CLM.BinaryOutDir     = False
run.Solver.CLM.IstepStart       = 1
run.Solver.WriteCLMBinary        = False
run.Solver.WriteSiloCLM          = False
run.Solver.CLM.WriteLogs         = False
run.Solver.CLM.WriteLastRST      = True
run.Solver.CLM.DailyRST          = False
run.Solver.CLM.SingleFile         = True

# ----- INITIAL CONDITIONS -----
# Hydrostatic equilibrium consistent with bottom BC
run.ICPressure.Type                  = "HydroStaticPatch"
run.ICPressure.GeomNames             = "domain"
run.Geom.domain.ICPressure.Value     = wtd_m
run.Geom.domain.ICPressure.RefGeom   = "domain"
run.Geom.domain.ICPressure.RefPatch  = "z_lower"

print("ParFlow run configured successfully")
print(f"  Physics: Medlyn stomata, CLM5Tanh interception, canopy clumping (via drv_vegp)")
print(f"  Soil: {CONUS2_SOILS[soil_type][0]} (Ksat={ksat}, porosity={porosity})")
print(f"  WTD BC: {wtd_m} m")

In [ ]:
# Run ParFlow
run.run(working_directory=static_write_dir, skip_validation=True)

## Quick-Look Output Verification

Load the NetCDF output and plot ET components and soil moisture to verify the run completed successfully.

In [ ]:
# Load CLM and ParFlow NetCDF output
import glob as glob_mod

run_dir = Path(static_write_dir)
clm_files = sorted(run_dir.glob("*.out.CLM.*.nc"))
pf_files = sorted(run_dir.glob("*.out.00001.nc"))

if not clm_files:
    raise FileNotFoundError(f"No CLM NetCDF output found in {run_dir}")

clm_ds = xr.open_dataset(clm_files[0])
print("CLM output variables:", list(clm_ds.data_vars))
print(f"  Time steps: {clm_ds.dims.get('time', 'N/A')}")

lh = clm_ds["eflx_lh_tot"].values.flatten()
print(f"  LH range: {lh.min():.1f} to {lh.max():.1f} W/m2, mean={lh.mean():.1f}")

if pf_files:
    pf_ds = xr.open_dataset(pf_files[0])
    print(f"\nParFlow output variables: {list(pf_ds.data_vars)}")
else:
    pf_ds = None
    print("\nNo ParFlow NetCDF output found (PFB output may be disabled)")

In [ ]:
# Plot ET components and soil moisture time series
dates = pd.date_range(start, periods=n_hours, freq="h")

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1: Latent heat flux
axes[0].plot(dates, clm_ds["eflx_lh_tot"].values.flatten(), lw=0.3, color="green")
axes[0].set_ylabel("LH Flux [W/m2]")
axes[0].set_title(f"{site_id} WY{water_year} — ParFlow-CLM Output")

# Panel 2: ET components (mm/day)
for var, label, color in [
    ("qflx_tran_veg", "Transpiration", "green"),
    ("qflx_evap_soi", "Soil Evaporation", "brown"),
    ("qflx_evap_veg", "Canopy Evaporation", "blue"),
]:
    if var in clm_ds:
        vals = clm_ds[var].values.flatten() * 3600 * 24  # mm/s -> mm/day
        axes[1].plot(dates, vals, lw=0.3, color=color, label=label)
axes[1].set_ylabel("Flux [mm/day]")
axes[1].legend(loc="upper right", fontsize=8)

# Panel 3: Soil moisture (top 2 layers)
if pf_ds is not None and "saturation" in pf_ds:
    sat = pf_ds["saturation"]
    axes[2].plot(dates, sat[:, 9, 0, 0].values * porosity * 100,
                 lw=0.5, color="steelblue", label="Top 10cm")
    axes[2].plot(dates, sat[:, 8, 0, 0].values * porosity * 100,
                 lw=0.5, color="navy", label="30cm")
    axes[2].set_ylabel("Vol. Water Content [%]")
    axes[2].legend(loc="upper right", fontsize=8)
else:
    axes[2].text(0.5, 0.5, "No saturation output", transform=axes[2].transAxes,
                 ha="center", va="center")

axes[2].set_xlabel("Date")
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.show()

clm_ds.close()
if pf_ds is not None:
    pf_ds.close()